# MorphoCause mechanistic pilot (v4, Qwen2.5-14B): a causal-role direction, and does it transfer to English?

**Runtime: A100** (the 14B needs about 30 GB in bf16). **Run all.** About 30 to 40 minutes including the download. Every result table is saved to Google Drive at the end (`MyDrive/morphocause_pilot/`). Approve the Drive prompt when it appears.

**The idea.** Unvocalised سبب X Y can mean "X caused Y" (verb reading) or "the cause of X is Y" (noun reading). When world knowledge picks the reading, the input is fixed but the interpretation is not, so an internal difference tracking which noun is the cause comes from interpretation, not from the tokens.

**What is new in v4**
- **Readout at the answer options.** The model reads left to right, so at the first noun inside the sentence it cannot yet know the role. Roles are read at "(A) X (B) Y", after the whole sentence; these tokens are unvocalised and identical in every condition.
- **Case check.** In all four سبب forms the cause carries the nominative (damma), so "cause" and "nominative" were confounded. v4 adds نَتَجَ Yُ عَنِ Xِ ("Y resulted from X"), where the effect is nominative. A direction that still finds the cause there tracks causal role; one that flips tracks case.
- **Convergence with word order held fixed:** unvocalised سبب X Y compared with سَبَّبَ Xُ Yَ and سَبَّبَ Xَ Yُ, which differ only in their case endings.
- **Results saved to Drive.**

**Inputs per pair (X is the real-world cause for familiar pairs)**

| Code | Sentence | Asserts |
|---|---|---|
| U_xy | سبب X Y | ambiguous; knowledge picks X → Y |
| U_yx | سبب Y X | ambiguous; noun reading gives X → Y |
| V_vso | سَبَّبَ Xُ Yَ | X → Y |
| V_vos | سَبَّبَ Xَ Yُ | Y → X |
| V_nom_xy | سَبَبُ Xِ Yُ | Y → X |
| V_nom_yx | سَبَبُ Yِ Xُ | X → Y |
| N_res | نَتَجَ Yُ عَنِ Xِ | X → Y, effect nominative |

Plus English versions and 8 fictional pairs with invented nouns.

**Go / nah, fixed in advance**
- G1: the model resolves at least 75% of unvocalised familiar items by world knowledge.
- G2: at the best middle layer, held-out accuracy at least 0.80, and at least 0.70 on pairs containing a role-switching word (fire, famine, disease, accident, poverty are causes in one pair and effects in another).
- G3: steering at strength 4 moves answers more than every random direction, flips at least 30% of fictional Arabic items, with random flips at most 10%.
- T: English read-out at least 0.75 and English neutral steering passes the G3 test.
- Ablation (section 9) and reverse transfer (section 10) have their own criteria, stated there.

Check the Arabic stimuli in the next cell before trusting anything.

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
%%writefile stimuli.py
# Familiar pairs: (Arabic cause, Arabic effect, English cause, English effect). Several entities switch roles across
# pairs (fire, famine, disease, accident, poverty), which separates causal role from word identity.
FAMILIAR = [
    ("التدخين", "السرطان", "smoking", "cancer"),
    ("الزلزال", "الدمار", "earthquake", "destruction"),
    ("المطر", "الفيضان", "rain", "flooding"),
    ("الحريق", "الدخان", "fire", "smoke"),
    ("البرق", "الحريق", "lightning", "fire"),
    ("الجفاف", "المجاعة", "drought", "famine"),
    ("المجاعة", "الهجرة", "famine", "migration"),
    ("الفيروس", "المرض", "virus", "disease"),
    ("المرض", "الوفاة", "disease", "death"),
    ("الإهمال", "الحادث", "negligence", "accident"),
    ("الحادث", "الإصابة", "accident", "injury"),
    ("الحرب", "الفقر", "war", "poverty"),
    ("الفقر", "الجريمة", "poverty", "crime"),
    ("الإجهاد", "الصداع", "stress", "headache"),
    ("التلوث", "الضباب", "pollution", "smog"),
]
# Fictional pairs: invented nouns with no meaning, in Arabic script and a matching English transliteration.
FICTIONAL = [
    ("الزرنوق", "البلطاس", "zarnuk", "baltas"),
    ("القمرود", "الشنفار", "qamrod", "shanfar"),
    ("الدلموس", "الفرطاق", "dalmus", "fartak"),
    ("الجعبوط", "الطرشين", "jaabut", "tarshin"),
    ("الكمبوس", "الغندوب", "kambus", "ghandub"),
    ("البرطوش", "السمعوط", "bartush", "samaut"),
    ("الدرنيف", "القلبوس", "darnif", "qalbus"),
    ("الحنطوب", "الزعموط", "hantub", "zaamut"),
]
DAMMA, FATHA, KASRA = "\u064F", "\u064E", "\u0650"
VERB_V, NOUN_V = "سَبَّبَ", "سَبَبُ"
RES_V, AN = "نَتَجَ", "عَنِ"

def arabic_inputs(x, y):
    """x, y: the pair's first and second noun as listed (for familiar pairs x is the real-world cause).
    Returns (kind, sentence, first noun, second noun, grammatical cause or None, vocalised?)."""
    return [
        ("U_xy",     f"سبب {x} {y}",                              x, y, None, False),
        ("U_yx",     f"سبب {y} {x}",                              y, x, None, False),
        ("V_vso",    f"{VERB_V} {x}{DAMMA} {y}{FATHA}",           x + DAMMA, y + FATHA, "x", True),  # x caused y
        ("V_vos",    f"{VERB_V} {x}{FATHA} {y}{DAMMA}",           x + FATHA, y + DAMMA, "y", True),  # y caused x
        ("V_nom_xy", f"{NOUN_V} {x}{KASRA} {y}{DAMMA}",           x + KASRA, y + DAMMA, "y", True),  # the cause of x is y
        ("V_nom_yx", f"{NOUN_V} {y}{KASRA} {x}{DAMMA}",           y + KASRA, x + DAMMA, "x", True),  # the cause of y is x
        # y resulted from x: the effect is nominative and the cause genitive, which breaks the link between
        # "cause" and "nominative" that holds in all four forms above
        ("N_res",    f"{RES_V} {y}{DAMMA} {AN} {x}{KASRA}",       y + DAMMA, x + KASRA, "x", True),
    ]

def english_inputs(x, y):
    """x is the cause in every template; positions and constructions vary."""
    return [
        ("E_active",  f"{x} caused {y}.",          x, y, "x"),
        ("E_nominal", f"The cause of {y} is {x}.", y, x, "x"),
        ("E_passive", f"{y} was caused by {x}.",   y, x, "x"),
    ]


In [ ]:
%%writefile mc_core.py
"""MorphoCause mechanistic pilot: analysis helpers (numpy only, so they can be tested without a GPU)."""
import numpy as np

SWITCHERS_AR = {"الحريق", "المجاعة", "المرض", "الحادث", "الفقر"}
# Which stored positions hold the two nouns for role readout: (0, 1) = the nouns inside the sentence,
# (3, 4) = the same nouns repeated in the answer options, after the model has read the whole sentence.
POS = (3, 4)

def last_token_overlapping(offsets, start, end):
    """Index of the last token whose character span overlaps [start, end)."""
    idx = [i for i, (s, e) in enumerate(offsets) if s < end and e > start and e > s]
    if not idx:
        raise ValueError(f"no token overlaps characters {start}-{end}")
    return max(idx)

def option_char_spans(text, opt_a, opt_b):
    """Character spans of the two answer options, '(A) ...' and '(B) ...', in the prompt."""
    ia, ib = text.rfind("(A) " + opt_a), text.rfind("(B) " + opt_b)
    if ia < 0 or ib < 0:
        raise ValueError("options not found in prompt")
    return (ia + 4, ia + 4 + len(opt_a)), (ib + 4, ib + 4 + len(opt_b))

def entity_char_spans(text, sentence, first, second):
    """Character spans of the sentence's two nouns inside the full prompt text, plus the closing-quote position."""
    q = text.find('"' + sentence + '"')
    if q < 0:
        raise ValueError("sentence not found in prompt")
    s0 = q + 1
    i1 = sentence.find(first)
    i2 = sentence.find(second, i1 + len(first))
    if i1 < 0 or i2 < 0:
        raise ValueError(f"nouns not found in sentence: {sentence}")
    end_quote = s0 + len(sentence)
    return (s0 + i1, s0 + i1 + len(first)), (s0 + i2, s0 + i2 + len(second)), (end_quote, end_quote + 1)

def unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

def role_diffs(H, recs, ref, layer):
    """H: [n, L+1, 5, d] states at (first noun, second noun, closing quote, option A noun, option B noun). ref(rec) -> 'first'/'second'/None.
    Returns cause-minus-effect vectors at `layer` for records with a defined cause."""
    out, keep = [], []
    for i, r in enumerate(recs):
        c = ref(r)
        if c is None:
            continue
        a, b = H[i, layer, POS[0]].astype(np.float32), H[i, layer, POS[1]].astype(np.float32)
        out.append(a - b if c == "first" else b - a); keep.append(i)
    return np.array(out), keep

def paired_acc(H, recs, ref, layer, u):
    d, keep = role_diffs(H, recs, ref, layer)
    if len(d) == 0:
        return np.nan, 0
    s = d @ u
    return float((s > 0).mean()), len(d)

def direction(H, recs, ref, layer):
    d, _ = role_diffs(H, recs, ref, layer)
    return unit(d.mean(0)), float((d @ unit(d.mean(0))).mean())

def lopo_acc(H, recs, train_mask, ref, layer, subset=None):
    """Leave-one-pair-out: for each pair, fit the direction on every other pair's training records,
    then score that pair's records. `subset(rec)` restricts which held-out records are scored."""
    pairs = sorted({r["pair"] for r, m in zip(recs, train_mask) if m})
    hits = []
    for p in pairs:
        tr = [i for i, (r, m) in enumerate(zip(recs, train_mask)) if m and r["pair"] != p]
        te = [i for i, (r, m) in enumerate(zip(recs, train_mask)) if m and r["pair"] == p and (subset is None or subset(r))]
        if not te:
            continue
        u, _ = direction(H[tr], [recs[i] for i in tr], ref, layer)
        d, _ = role_diffs(H[te], [recs[i] for i in te], ref, layer)
        hits += list(d @ u > 0)
    return (float(np.mean(hits)) if hits else np.nan), len(hits)

def centred_cos(a, b, mu):
    a, b = a - mu, b - mu
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))


## 1. Load the model

In [ ]:
import json, math, random, gc
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
import importlib, stimuli, mc_core as m
importlib.reload(stimuli); importlib.reload(m)
from stimuli import FAMILIAR, FICTIONAL, arabic_inputs, english_inputs

MODEL = "Qwen/Qwen2.5-14B-Instruct"    # about 30 GB in bf16: needs an A100 (use the 7B on an L4)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map={"": 0}).eval()
L = model.config.num_hidden_layers
A_ID = tok.encode("A", add_special_tokens=False)[0]
B_ID = tok.encode("B", add_special_tokens=False)[0]
print(MODEL, "|", L, "layers | answer tokens:", repr(tok.decode([A_ID])), repr(tok.decode([B_ID])))

## 2. Probe every item
Both answer orders per item. Hidden states at every layer at five positions: the two nouns in the sentence, the closing quote, and the two answer-option nouns.

In [ ]:
def prompt_text(sentence, o1, o2, lang):
    user = (f'Read this {lang} sentence: "{sentence}"\nAccording to the sentence, which is the cause?\n'
            f'(A) {o1}\n(B) {o2}\nAnswer with A or B.')
    return tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)

def encode(text, sentence, first, second, opt_a, opt_b):
    """Token positions: [first noun, second noun, closing quote, option A noun, option B noun]."""
    enc = tok(text, return_tensors="pt", return_offsets_mapping=True, add_special_tokens=False)
    offs = enc["offset_mapping"][0].tolist()
    s1, s2, sq = m.entity_char_spans(text, sentence, first, second)
    sa, sb = m.option_char_spans(text, opt_a, opt_b)
    pos = [m.last_token_overlapping(offs, *s) for s in (s1, s2, sq, sa, sb)]
    return enc["input_ids"].to(model.device), pos

@torch.no_grad()
def ab_logit(ids, steer=None):
    """log P(A) - log P(B) at the answer position. steer = (hs_index, [(position, vector), ...]) adds vectors
    to the residual stream after that layer; hidden_states[l] is the output of decoder layer l - 1."""
    handle = None
    if steer is not None:
        layer, edits = steer
        def hook(mod, inp, out):
            hs = out[0] if isinstance(out, tuple) else out
            for p, v in edits:
                hs[:, p, :] += v
            return out
        handle = model.model.layers[layer - 1].register_forward_hook(hook)
    try:
        lp = torch.log_softmax(model(input_ids=ids).logits[0, -1].float(), -1)
    finally:
        if handle is not None:
            handle.remove()
    return (lp[A_ID] - lp[B_ID]).item()

@torch.no_grad()
def probe_item(sentence, first, second, opt_first, opt_second, lang, steer_fn=None):
    """Returns (score, states). score > 0 means the model picks the first noun as the cause; it averages both
    option orders so a preference for answering A cancels out. states: [L+1, 5, d] at the two nouns in the
    sentence, the closing quote, and the two option nouns (A = first noun), from the first option order.
    Steering edits land on the option nouns in each order."""
    t1, t2 = prompt_text(sentence, opt_first, opt_second, lang), prompt_text(sentence, opt_second, opt_first, lang)
    ids1, pos = encode(t1, sentence, first, second, opt_first, opt_second)
    ids2, pos2 = encode(t2, sentence, first, second, opt_second, opt_first)
    assert pos[:3] == pos2[:3]
    if steer_fn is None:
        out = model(input_ids=ids1, output_hidden_states=True)
        lp = torch.log_softmax(out.logits[0, -1].float(), -1); d1 = (lp[A_ID] - lp[B_ID]).item()
        H = torch.stack([h[0, pos, :] for h in out.hidden_states]).float().cpu().numpy().astype(np.float16)
        d2 = ab_logit(ids2)
        return (d1 - d2) / 2, H
    # pos2 lists option A then B, which in the second order are (second noun, first noun): swap back
    pos2_aligned = pos2[:3] + [pos2[4], pos2[3]]
    return (ab_logit(ids1, steer_fn(pos)) - ab_logit(ids2, steer_fn(pos2_aligned))) / 2, None

SWITCH = {"الحريق", "المجاعة", "المرض", "الحادث", "الفقر"}
recs, H = [], []
for setname, pairs in (("familiar", FAMILIAR), ("fictional", FICTIONAL)):
    for pid, (x, y, ex, ey) in enumerate(pairs):
        for kind, s, a, b, g, voc in arabic_inputs(x, y):
            first_is_x = a.startswith(x)
            gram = None if g is None else ("first" if (g == "x") == first_is_x else "second")
            o1, o2 = (x, y) if first_is_x else (y, x)
            sc, h = probe_item(s, a, b, o1, o2, "Arabic")
            recs.append(dict(pair=f"{setname}{pid}", set=setname, lang="ar", kind=kind, sentence=s, first=o1, second=o2,
                             first_tok=a, second_tok=b, first_is_x=first_is_x, gram=gram, score=sc,
                             switch=bool({x, y} & SWITCH)))
            H.append(h)
        for kind, s, a, b, g in english_inputs(ex, ey):
            first_is_x = a == ex
            sc, h = probe_item(s, a, b, a, b, "English")
            recs.append(dict(pair=f"{setname}{pid}", set=setname, lang="en", kind=kind, sentence=s, first=a, second=b,
                             first_is_x=first_is_x, gram="first" if first_is_x else "second", score=sc, switch=False))
            H.append(h)
H = np.stack(H)
df = pd.DataFrame(recs)
print(len(recs), "items probed | states:", H.shape)

## 3. Behaviour

In [ ]:
know = lambda r: ("first" if r["first_is_x"] else "second") if r["set"] == "familiar" else None
gramr = lambda r: r["gram"]
chosen = lambda r: None if abs(r["score"]) < 0.5 else ("first" if r["score"] > 0 else "second")
picks = lambda r, ref: (r["score"] > 0) == (ref(r) == "first")

def rate(sel, ref):
    rs = [r for r in recs if sel(r) and ref(r) is not None]
    return f"{sum(picks(r, ref) for r in rs)}/{len(rs)}", (np.mean([picks(r, ref) for r in rs]) if rs else np.nan)

rows = [
 ("Arabic unvocalised, familiar", "picks the real-world cause", *rate(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"].startswith("U"), know)),
 ("Arabic vocalised, familiar, grammar agrees with knowledge", "follows grammar", *rate(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"] in ("V_vso", "V_nom_yx"), gramr)),
 ("Arabic vocalised, familiar, grammar contradicts knowledge", "follows grammar", *rate(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"] in ("V_vos", "V_nom_xy"), gramr)),
 ("Arabic unvocalised, fictional", "picks the first noun", *rate(lambda r: r["lang"] == "ar" and r["set"] == "fictional" and r["kind"].startswith("U"), lambda r: "first")),
 ("Arabic vocalised, fictional", "follows grammar", *rate(lambda r: r["lang"] == "ar" and r["set"] == "fictional" and r["kind"].startswith("V"), gramr)),
 ("Arabic 'Y resulted from X' (effect nominative)", "follows grammar", *rate(lambda r: r["lang"] == "ar" and r["kind"] == "N_res", gramr)),
 ("English, familiar", "correct", *rate(lambda r: r["lang"] == "en" and r["set"] == "familiar", gramr)),
 ("English, fictional", "correct", *rate(lambda r: r["lang"] == "en" and r["set"] == "fictional", gramr)),
]
beh = pd.DataFrame(rows, columns=["inputs", "measure", "count", "rate"]).set_index("inputs")
display(beh.round(2))
G1 = beh.loc["Arabic unvocalised, familiar", "rate"] >= 0.75
print("G1, the model resolves the unvocalised sentences by world knowledge (>= 75%):", "PASS" if G1 else "FAIL")

## 4. Causal-role direction across layers (read at the answer options)
Thick lines are the main test. Lines near 0 rather than 0.5 mean the direction encodes word identity, not role. The 'resulted from' line is the case check.

In [ ]:
is_ar_unvoc_fam = [r["lang"] == "ar" and r["set"] == "familiar" and r["kind"].startswith("U") for r in recs]
tr_idx = [i for i, ok in enumerate(is_ar_unvoc_fam) if ok]
def subset_idx(sel): return [i for i, r in enumerate(recs) if sel(r)]
sets = {
 "fictional Arabic, unvocalised (vs the model's own choice)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "fictional" and r["kind"].startswith("U")), chosen),
 "Arabic vocalised (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["kind"].startswith("V")), gramr),
 "Arabic, grammar contradicts knowledge (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"] in ("V_vos", "V_nom_xy")), gramr),
 "Arabic 'resulted from', effect nominative (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["kind"] == "N_res"), gramr),
 "English familiar (vs truth)": (subset_idx(lambda r: r["lang"] == "en" and r["set"] == "familiar"), gramr),
 "English fictional (vs truth)": (subset_idx(lambda r: r["lang"] == "en" and r["set"] == "fictional"), gramr),
}
curves = {"Arabic unvocalised familiar, held-out pairs": [], "  same, pairs with a role-switching word": []}
curves.update({k: [] for k in sets}); gaps = []
for l in range(L + 1):
    curves["Arabic unvocalised familiar, held-out pairs"].append(m.lopo_acc(H, recs, is_ar_unvoc_fam, know, l)[0])
    curves["  same, pairs with a role-switching word"].append(m.lopo_acc(H, recs, is_ar_unvoc_fam, know, l, subset=lambda r: r["switch"])[0])
    u, gap = m.direction(H[tr_idx], [recs[i] for i in tr_idx], know, l); gaps.append(gap)
    for k, (idx, ref) in sets.items():
        curves[k].append(m.paired_acc(H[idx], [recs[i] for i in idx], ref, l, u)[0])
lay = pd.DataFrame(curves)
fig, ax = plt.subplots(figsize=(10, 5))
for k in lay.columns:
    ax.plot(lay.index, lay[k], label=k, lw=2.2 if k.startswith("Arabic unvocalised") else 1.3)
ax.axhline(0.5, color="grey", ls=":", lw=1); ax.set_ylim(0, 1.02)
ax.set_xlabel("layer"); ax.set_ylabel("cause scored above effect (fraction of items)")
ax.set_title(f"Causal-role direction at the {'answer options' if m.POS == (3, 4) else 'nouns in the sentence'}, fitted on unvocalised familiar Arabic"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()
lo, hi = int(0.25 * L), int(0.85 * L)
LSTAR = int(lo + np.argmax(lay["Arabic unvocalised familiar, held-out pairs"].values[lo:hi + 1]))
display(lay.loc[[LSTAR]].T.rename(columns={LSTAR: f"layer {LSTAR}"}).round(2))
G2 = (lay.loc[LSTAR, "Arabic unvocalised familiar, held-out pairs"] >= 0.80 and
      lay.loc[LSTAR, "  same, pairs with a role-switching word"] >= 0.70)
print(f"Chosen layer: {LSTAR} (best held-out accuracy between layers {lo} and {hi})")
print("G2, a role direction beyond word identity (held-out >= 0.80 and role-switching pairs >= 0.70):", "PASS" if G2 else "FAIL")

## 5. Within-word test

In [ ]:
# Within-word test: each role-switching word is a cause in one pair and an effect in another.
# The direction is refitted without either pair, then the word's own projection is compared across its two roles.
res = []
for wd in sorted(SWITCH):
    keep = [i for i in tr_idx if wd not in (recs[i]["first"], recs[i]["second"])]
    u, _ = m.direction(H[keep], [recs[i] for i in keep], know, LSTAR)
    as_cause, as_effect = [], []
    for i in tr_idx:
        r = recs[i]
        if wd not in (r["first"], r["second"]): continue
        k = 0 if r["first"] == wd else 1
        p = float(H[i, LSTAR, m.POS[k]].astype(np.float32) @ u)
        is_cause = (know(r) == "first") == (k == 0)
        (as_cause if is_cause else as_effect).append(p)
    res.append(dict(word=wd, as_cause=np.mean(as_cause), as_effect=np.mean(as_effect),
                    role_tracked=np.mean(as_cause) > np.mean(as_effect)))
words = pd.DataFrame(res); display(words.round(2))
print(f"Role-switching words scored higher as cause than as effect: {int(words.role_tracked.sum())}/{len(words)}")

## 6. Convergence, word order held fixed

In [ ]:
# Convergence at the closing quote, with word order held fixed: is unvocalised "سبب X Y" closer to
# "X caused Y" (V_vso) than to "Y caused X" (V_vos)? The two vocalised forms differ only in their case endings.
# Familiar pairs should lean towards the reading world knowledge picks; fictional pairs have nothing to lean on.
by = {(r["pair"], r["kind"]): i for i, r in enumerate(recs) if r["lang"] == "ar"}
ar_idx = [i for i, r in enumerate(recs) if r["lang"] == "ar"]
conv = {"familiar": [], "fictional": []}; conv_sd = {"familiar": [], "fictional": []}
for l in range(L + 1):
    mu = H[ar_idx, l, 2].astype(np.float32).mean(0)
    V = lambda key: H[by[key], l, 2].astype(np.float32)
    for setname in conv:
        ds = [m.centred_cos(V((p, "U_xy")), V((p, "V_vso")), mu) - m.centred_cos(V((p, "U_xy")), V((p, "V_vos")), mu)
              for p in sorted({r["pair"] for r in recs if r["set"] == setname})]
        conv[setname].append(np.mean(ds)); conv_sd[setname].append(np.std(ds) / np.sqrt(len(ds)))
fig, ax = plt.subplots(figsize=(10, 4))
for k in conv:
    v, se = np.array(conv[k]), np.array(conv_sd[k])
    ax.plot(range(L + 1), v, label=k, lw=2); ax.fill_between(range(L + 1), v - 2 * se, v + 2 * se, alpha=0.15)
ax.axhline(0, color="grey", ls=":", lw=1); ax.set_xlabel("layer")
ax.set_ylabel("closer to 'X caused Y'\\nthan to 'Y caused X'")
ax.set_title("Convergence with word order held fixed (bands: about 95% of the mean)"); ax.legend()
plt.tight_layout(); plt.show()
CONV_GAP = conv["familiar"][LSTAR] - conv["fictional"][LSTAR]
print(f"At layer {LSTAR}: familiar {conv['familiar'][LSTAR]:+.3f} (se {conv_sd['familiar'][LSTAR]:.3f}), "
      f"fictional {conv['fictional'][LSTAR]:+.3f} (se {conv_sd['fictional'][LSTAR]:.3f})")

## 7. Steering at the answer options

In [ ]:
# Causal test: push the role direction onto the noun the model did NOT pick as cause (and away from the one it did)
# at the chosen layer, and see whether its answer moves. Random directions of the same size are the control.
u, gap = m.direction(H[tr_idx], [recs[i] for i in tr_idx], know, LSTAR)
U_T = torch.tensor(u, dtype=torch.bfloat16, device=model.device)
rng = np.random.default_rng(1234)
RAND = [torch.tensor(m.unit(rng.normal(size=u.shape[0])), dtype=torch.bfloat16, device=model.device) for _ in range(5)]
ALPHAS = [1, 2, 4, 8]

steer_sets = {
 "Arabic fictional, unvocalised": [r for r in recs if r["lang"] == "ar" and r["set"] == "fictional" and r["kind"].startswith("U")],
 "Arabic familiar, unvocalised (against knowledge)": [r for r in recs if r["lang"] == "ar" and r["set"] == "familiar" and r["kind"].startswith("U")],
}
neutral = []   # English sentences with no causal grammar at all
for setname, pairs in (("fictional", FICTIONAL), ("familiar", FAMILIAR)):
    for pid, (x, y, ex, ey) in enumerate(pairs):
        s = f"There is a causal link between {ex} and {ey}."
        sc, _ = probe_item(s, ex, ey, ex, ey, "English")
        neutral.append(dict(set=setname, lang="en", sentence=s, first=ex, second=ey, score=sc))
steer_sets["English neutral, fictional (transfer)"] = [r for r in neutral if r["set"] == "fictional"]
steer_sets["English neutral, familiar (transfer, against knowledge)"] = [r for r in neutral if r["set"] == "familiar"]

def run_steer(r, vec, alpha):
    target_first = r["score"] < 0                    # push the noun the model did not choose
    def steer_fn(pos):
        v = vec * (alpha * gap / 2)
        return (LSTAR, [(pos[m.POS[0]], v if target_first else -v), (pos[m.POS[1]], -v if target_first else v)])
    lang = "Arabic" if r["lang"] == "ar" else "English"
    sc, _ = probe_item(r["sentence"], r["first"], r["second"], r["first"], r["second"], lang, steer_fn)
    toward = (sc - r["score"]) if target_first else (r["score"] - sc)
    flipped = (sc > 0) == target_first
    return toward, flipped

out = []
for name, items in steer_sets.items():
    for alpha in ALPHAS:
        real = [run_steer(r, U_T, alpha) for r in items]
        rand = [[run_steer(r, R, alpha) for r in items] for R in RAND]
        out.append(dict(set=name, alpha=alpha, n=len(items),
                        shift_real=np.mean([t for t, _ in real]), flip_real=np.mean([f for _, f in real]),
                        shift_rand_max=max(np.mean([t for t, _ in rr]) for rr in rand),
                        flip_rand_mean=np.mean([np.mean([f for _, f in rr]) for rr in rand])))
steer = pd.DataFrame(out); display(steer.round(3))

def steer_pass(name, alpha=4):
    row = steer[(steer.set == name) & (steer.alpha == alpha)].iloc[0]
    return row.shift_real > row.shift_rand_max and row.flip_real >= 0.30 and row.flip_rand_mean <= 0.10
G3 = steer_pass("Arabic fictional, unvocalised")
T_steer = steer_pass("English neutral, fictional (transfer)")
print("G3, steering moves Arabic answers beyond random directions (alpha 4):", "PASS" if G3 else "FAIL")

## 8. Verdict

In [ ]:
T_probe = lay.loc[LSTAR, "English familiar (vs truth)"] >= 0.75
T = bool(T_probe and T_steer)
print("Summary at layer", LSTAR)
print(f"  G1 model resolves Arabic by knowledge ........ {'PASS' if G1 else 'FAIL'}  ({beh.loc['Arabic unvocalised, familiar', 'count']})")
print(f"  G2 role direction beyond word identity ....... {'PASS' if G2 else 'FAIL'}")
print(f"  G3 steering changes Arabic answers ........... {'PASS' if G3 else 'FAIL'}")
print(f"  T  transfers to English (probe and steering) . {'PASS' if T else 'FAIL'}  (probe {'pass' if T_probe else 'fail'}, steering {'pass' if T_steer else 'fail'})")
print(f"  role-switching words tracked ................. {int(words.role_tracked.sum())}/{len(words)}")
print(f"  convergence, familiar minus fictional ........ {CONV_GAP:+.3f}  (word order held fixed)")
case_acc = lay.loc[LSTAR, "Arabic 'resulted from', effect nominative (vs grammar)"]
case_note = ("tracks the CAUSE, not nominative case" if case_acc >= 0.75 else
             "tracks NOMINATIVE CASE, not the cause" if case_acc <= 0.25 else "mixed between cause and case")
print(f"  case check, 'Y resulted from X' .............. {case_acc:.2f}  -> {case_note}")
if not G1:
    verdict = "NAH (instrument): the model does not resolve the Arabic sentences by world knowledge; try a stronger Arabic model first."
elif not G2:
    verdict = "NAH: no causal-role direction beyond word identity in the middle layers."
elif not G3:
    verdict = "PARTIAL: a readable role direction exists, but steering along it does not change answers (readable, not used)."
elif not T:
    verdict = "GO: a causal-role direction the model reads and uses in Arabic. English transfer not shown yet."
else:
    verdict = "GO, EXTREMELY INTERESTING: the Arabic role direction is read, causally used, and transfers to English."
print("\nVERDICT:", verdict)
print("\nPilot caveats: 15 familiar and 8 fictional pairs, one model, the layer chosen on the same data; "
      "native-speaker checks of the stimuli are still needed before any claim.")

## 9. Ablation: is the direction used?
Projects the role direction (rank 1) or its top 4 components (rank 4) out of every position across the middle-to-late layers (25 to 38 in the 14B), and compares with random subspaces of the same rank. No strength setting to get wrong.

**Pass (fixed before running):** the role ablation hurts more than every random ablation, and it either halves the answer margin, or costs at least 15 points of accuracy while random ablations cost at most 5.

In [ ]:
# ===== 9. Ablation: remove the role direction across the middle-to-late layers (15-22 in Qwen2.5-7B) and see if behaviour breaks =====
# No strength knob: the direction is projected out of every position. Random subspaces of the same rank are the control.
BAND = list(range(round(0.53 * L), round(0.79 * L) + 1))   # layers 15-22 for a 28-layer model; layer l edits decoder layer l - 1
RANKS = [1, 4]                      # rank 1 = the mean direction; rank 4 = top 4 components of cause-minus-effect
N_RAND = 5

def role_basis(l, k):
    D, _ = m.role_diffs(H[tr_idx], [recs[i] for i in tr_idx], know, l)
    Vt = np.linalg.svd(D, full_matrices=False)[2]
    B = Vt[:k] if k > 1 else m.unit(D.mean(0))[None]
    return torch.tensor(B, dtype=torch.bfloat16, device=model.device)

def random_basis(k, seed):
    q, _ = np.linalg.qr(np.random.default_rng(seed).normal(size=(H.shape[-1], k)))
    return torch.tensor(q.T, dtype=torch.bfloat16, device=model.device)

@torch.no_grad()
def score_item(r, basis=None):
    """Answer score (> 0 means the first noun is the cause) with an optional {layer: basis} projected out."""
    lang = "Arabic" if r["lang"] == "ar" else "English"
    t1, t2 = prompt_text(r["sentence"], r["first"], r["second"], lang), prompt_text(r["sentence"], r["second"], r["first"], lang)
    a1, b1 = r.get("first_tok", r["first"]), r.get("second_tok", r["second"])
    ids1, _ = encode(t1, r["sentence"], a1, b1, r["first"], r["second"])
    ids2, _ = encode(t2, r["sentence"], a1, b1, r["second"], r["first"])
    handles = []
    for l, B in (basis or {}).items():
        def hook(mod, inp, out, B=B):
            hs = out[0] if isinstance(out, tuple) else out
            hs -= (hs @ B.T) @ B
            return out
        handles.append(model.model.layers[l - 1].register_forward_hook(hook))
    try:
        return (ab_logit(ids1) - ab_logit(ids2)) / 2
    finally:
        for h in handles: h.remove()

def first_tokens(r):
    # surface forms of the nouns (with any case endings) were stored when the items were probed
    return r

abl_sets = {
 "Arabic unvocalised familiar (vs knowledge)": ([first_tokens(r) for r in recs if r["lang"] == "ar" and r["set"] == "familiar" and r["kind"].startswith("U")], know),
 "Arabic vocalised fictional (vs grammar)": ([first_tokens(r) for r in recs if r["lang"] == "ar" and r["set"] == "fictional" and (r["kind"].startswith("V") or r["kind"] == "N_res")], gramr),
 "English fictional (vs grammar)": ([r for r in recs if r["lang"] == "en" and r["set"] == "fictional"], gramr),
}
def summarise_scores(items, ref, scores):
    sign = np.array([1 if ref(r) == "first" else -1 for r in items])
    s = np.array(scores) * sign
    return float((s > 0).mean()), float(s.mean())

conds = {"baseline": None}
for k in RANKS:
    conds[f"role, rank {k}"] = {l: role_basis(l, k) for l in BAND}
    for j in range(N_RAND):
        conds[f"random {j}, rank {k}"] = {l: random_basis(k, 100 * k + 10 * l + j) for l in BAND}
rows_a = []
for name, (items, ref) in abl_sets.items():
    for cname, basis in conds.items():
        acc, margin = summarise_scores(items, ref, [score_item(r, basis) for r in items])
        rows_a.append(dict(items=name, condition=cname, accuracy=acc, margin=margin))
abl = pd.DataFrame(rows_a)

summary_rows = []
for name in abl_sets:
    g = abl[abl["items"] == name]; base = g[g.condition == "baseline"].iloc[0]
    for k in RANKS:
        real = g[g.condition == f"role, rank {k}"].iloc[0]
        rnd = g[g.condition.str.startswith("random") & g.condition.str.endswith(f"rank {k}")]
        summary_rows.append(dict(items=name, rank=k, base_acc=base.accuracy, role_acc=real.accuracy,
                                 random_acc_mean=rnd.accuracy.mean(), base_margin=base.margin, role_margin=real.margin,
                                 random_margin_min=rnd.margin.min()))
abl_sum = pd.DataFrame(summary_rows); display(abl_sum.round(3))

def ablation_pass(name):
    """Pass at some rank: the role ablation hurts more than every random ablation, AND either it at least halves the
    margin or it costs at least 15 points of accuracy while random ablations cost at most 5."""
    for k in RANKS:
        r = abl_sum[(abl_sum["items"] == name) & (abl_sum["rank"] == k)].iloc[0]
        beats_random = r.role_margin < r.random_margin_min
        big = (r.base_margin - r.role_margin >= 0.5 * r.base_margin) or \
              (r.base_acc - r.role_acc >= 0.15 and r.base_acc - r.random_acc_mean <= 0.05)
        if beats_random and big:
            return True, k
    return False, None
G3b, k_used = ablation_pass("Arabic unvocalised familiar (vs knowledge)")
G3b_en, _ = ablation_pass("English fictional (vs grammar)")
print("G3b, ablating the role direction breaks knowledge-based Arabic answers beyond random ablation:",
      f"PASS (rank {k_used})" if G3b else "FAIL")
print("Same test on English fictional (grammar-only roles):", "PASS" if G3b_en else "FAIL")

## 10. Reverse transfer: English role direction, tested on Arabic
Fitted on English sentences with invented nouns (24 pairs, three constructions each), where grammar alone fixes the roles and no word meaning can leak. Tested on the Arabic items, including the role-switching words.

**Pass (fixed before running):** at the best middle layer, at least 0.75 on unvocalised familiar Arabic, at least 0.65 on pairs with a role-switching word, and at least 4 of 5 switching words scored higher as cause than as effect.

In [ ]:
# ===== 10. Reverse transfer: fit the role direction on English sentences with invented nouns, test it on Arabic =====
# English with invented nouns has explicit grammar and no word meaning to leak, so its role direction is clean.
EXTRA_EN = [("plovex", "dranith"), ("quorvel", "skelmo"), ("vantrel", "morquin"), ("telbrisk", "zundar"),
            ("haxim", "prolvet"), ("drebin", "kesmar"), ("fyntor", "gorvane"), ("nelpix", "stravon"),
            ("brexil", "wombral"), ("tazique", "ollimar"), ("crendo", "yarvex"), ("juthric", "pelmont"),
            ("sorvig", "klandor"), ("mivrel", "threnox"), ("basquin", "ulvrent"), ("rontiz", "eskaril")]
en_recs, en_H = [], []
for pid, (x, y) in enumerate([(ex, ey) for _, _, ex, ey in FICTIONAL] + EXTRA_EN):
    for kind, s, a, b, g in english_inputs(x, y):
        first_is_x = a == x
        sc, h = probe_item(s, a, b, a, b, "English")
        en_recs.append(dict(pair=f"enfic{pid}", set="fictional", lang="en", kind=kind, sentence=s, first=a, second=b,
                            first_is_x=first_is_x, gram="first" if first_is_x else "second", score=sc, switch=False))
        en_H.append(h)
en_H = np.stack(en_H)
print(len(en_recs), "English sentences with invented nouns")

tests = {
 "Arabic unvocalised familiar (vs knowledge)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"].startswith("U")), know),
 "  same, pairs with a role-switching word": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"].startswith("U") and r["switch"]), know),
 "Arabic unvocalised fictional (vs the model's choice)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "fictional" and r["kind"].startswith("U")), chosen),
 "Arabic vocalised fictional (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "fictional" and r["kind"].startswith("V")), gramr),
 "Arabic, grammar contradicts knowledge (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"] in ("V_vos", "V_nom_xy")), gramr),
 "Arabic 'resulted from', effect nominative (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["kind"] == "N_res"), gramr),
}
rev = {"English invented nouns, held-out pairs (in-domain check)": []}; rev.update({k: [] for k in tests})
en_mask = [True] * len(en_recs)
for l in range(L + 1):
    rev["English invented nouns, held-out pairs (in-domain check)"].append(m.lopo_acc(en_H, en_recs, en_mask, gramr, l)[0])
    u_en, _ = m.direction(en_H, en_recs, gramr, l)
    for k, (idx, ref) in tests.items():
        rev[k].append(m.paired_acc(H[idx], [recs[i] for i in idx], ref, l, u_en)[0])
rv = pd.DataFrame(rev)
fig, ax = plt.subplots(figsize=(10, 5))
for k in rv.columns:
    ax.plot(rv.index, rv[k], label=k, lw=2.2 if k.startswith("Arabic unvocalised familiar") else 1.3)
ax.axhline(0.5, color="grey", ls=":", lw=1); ax.set_ylim(0, 1.02); ax.set_xlabel("layer")
ax.set_ylabel("cause scored above effect (fraction of items)")
ax.set_title("Role direction fitted on English with invented nouns, tested on Arabic"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); plt.show()

lo, hi = int(0.25 * L), int(0.85 * L)
main = "Arabic unvocalised familiar (vs knowledge)"
LREV = int(lo + np.argmax(rv[main].values[lo:hi + 1]))
display(rv.loc[[LREV]].T.rename(columns={LREV: f"layer {LREV}"}).round(2))

u_en, _ = m.direction(en_H, en_recs, gramr, LREV)
res_w = []
for wd in sorted(SWITCH):
    as_c, as_e = [], []
    for i in tests[main][0]:
        r = recs[i]
        if wd not in (r["first"], r["second"]): continue
        k = 0 if r["first"] == wd else 1
        p = float(H[i, LREV, m.POS[k]].astype(np.float32) @ u_en)
        (as_c if (know(r) == "first") == (k == 0) else as_e).append(p)
    res_w.append(dict(word=wd, as_cause=np.mean(as_c), as_effect=np.mean(as_e), role_tracked=np.mean(as_c) > np.mean(as_e)))
words_en = pd.DataFrame(res_w); display(words_en.round(2))
R = (rv.loc[LREV, main] >= 0.75 and rv.loc[LREV, "  same, pairs with a role-switching word"] >= 0.65
     and words_en.role_tracked.sum() >= 4)
print(f"Layer {LREV} (best between {lo} and {hi}).")
print("R, the English-derived role direction reads Arabic unvocalised roles beyond word identity:", "PASS" if R else "FAIL")

## 11. Follow-up verdict

In [ ]:
print("Follow-up summary")
print(f"  G3b ablation breaks Arabic knowledge-based answers .. {'PASS' if G3b else 'FAIL'}")
print(f"  ablation also breaks English invented-noun answers .. {'PASS' if G3b_en else 'FAIL'}")
print(f"  R   English-derived direction reads Arabic roles .... {'PASS' if R else 'FAIL'}  (layer {LREV})")
print(f"  role-switching words tracked by the English direction: {int(words_en.role_tracked.sum())}/{len(words_en)}")
if R and G3b:
    v = "GO, EXTREMELY INTERESTING: a shared causal-role representation, read across languages and used by the model."
elif R:
    v = "GO (readout): English-derived role reads ambiguous Arabic beyond word identity; causal use not shown yet."
elif G3b:
    v = "GO (use): removing the direction breaks Arabic answers, but the shared cross-lingual readout is not shown."
else:
    v = "NAH at 7B: no shared role representation detectable here. Try the 14B before dropping the idea."
print("\nFOLLOW-UP VERDICT:", v)
print("Caveats: small item counts, one model, layers chosen on the same data. Treat any GO as a reason to scale up, not a finding.")

## 12. Save everything to Drive

In [ ]:
# Save every result table (and the plots' data) to Google Drive, so nothing is lost if the runtime resets.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    out_dir = f"/content/drive/MyDrive/morphocause_pilot/{MODEL.split('/')[-1]}"
except Exception as e:
    print("Drive not available, saving locally:", repr(e)[:80]); out_dir = f"morphocause_pilot/{MODEL.split('/')[-1]}"
os.makedirs(out_dir, exist_ok=True)
tables = {"behaviour": beh, "role_by_layer": lay, "within_word": words, "steering": steer,
          "convergence": pd.DataFrame(conv), "ablation": globals().get("abl_sum"), "reverse_by_layer": globals().get("rv"),
          "reverse_within_word": globals().get("words_en"), "items": df.drop(columns=[], errors="ignore")}
for name, t in tables.items():
    if t is not None:
        t.to_csv(f"{out_dir}/{name}.csv")
print("saved to", out_dir, ":", sorted(os.listdir(out_dir)))

## 13. Control: causal role, or just "pick this option"?
Section 7 steered with the question "which is the cause?", so a direction meaning "pick this option" would also pass. Here the same steering runs under three questions:

| Question | If the direction is causal role | If it is answer selection |
|---|---|---|
| Which is the cause? | pushed noun picked more | pushed noun picked more |
| Which is the effect? | the **other** noun picked more | pushed noun picked more |
| Which word comes first? | no change | pushed noun picked more |

**Verdict rule (fixed before running), at strength 4:** CAUSAL ROLE if the effect-question shift is negative and at least half the size of the cause-question shift, and the word-order shift is under a quarter of it. ANSWER SELECTION if both the effect and word-order shifts are positive and at least half the cause shift. Otherwise MIXED.

In [ ]:
# ===== 13. Control: causal role, or just "pick this option"? =====
# Same steering as section 7 (push "cause" onto the noun the model did not pick), but under three questions:
#   cause  - "which is the cause?"      role or answer-selection: the pushed noun is picked more
#   effect - "which is the effect?"     ROLE: the OTHER noun is picked more   | ANSWER-SELECTION: the pushed noun
#   first  - "which word comes first?"  ROLE: no change                         | ANSWER-SELECTION: the pushed noun
QUESTIONS = {"cause": "According to the sentence, which is the cause?",
             "effect": "According to the sentence, which is the effect?",
             "first": "Which of these two words appears first in the sentence?"}
CTRL_ALPHAS, N_RAND_CTRL = [2, 4], 3

def prompt_q(sentence, o1, o2, lang, q):
    user = (f'Read this {lang} sentence: "{sentence}"\n{QUESTIONS[q]}\n'
            f'(A) {o1}\n(B) {o2}\nAnswer with A or B.')
    return tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def probe_q(r, q, steer_fn=None):
    """Score > 0: the model picks the first noun. Averages both option orders; steering edits the option nouns."""
    lang = "Arabic" if r["lang"] == "ar" else "English"
    a, b = r.get("first_tok", r["first"]), r.get("second_tok", r["second"])
    t1, t2 = prompt_q(r["sentence"], r["first"], r["second"], lang, q), prompt_q(r["sentence"], r["second"], r["first"], lang, q)
    ids1, p1 = encode(t1, r["sentence"], a, b, r["first"], r["second"])
    ids2, p2 = encode(t2, r["sentence"], a, b, r["second"], r["first"])
    p2 = p2[:3] + [p2[4], p2[3]]
    s1 = steer_fn(p1) if steer_fn else None
    s2 = steer_fn(p2) if steer_fn else None
    return (ab_logit(ids1, s1) - ab_logit(ids2, s2)) / 2

ctrl_sets = {k: v for k, v in steer_sets.items()}       # same items as section 7
rows_c = []
for name, items in ctrl_sets.items():
    for r in items:
        target_first = r["score"] < 0                     # the noun the model did NOT pick as cause
        sgn = 1 if target_first else -1                   # converts scores to "towards the pushed noun"
        base = {q: probe_q(r, q) for q in QUESTIONS}
        for alpha in CTRL_ALPHAS:
            for kind, vecs in (("role", [U_T]), ("random", RAND[:N_RAND_CTRL])):
                for vi, vec in enumerate(vecs):
                    def steer_fn(pos, vec=vec, alpha=alpha):
                        v = vec * (alpha * gap / 2)
                        return (LSTAR, [(pos[m.POS[0]], v if target_first else -v), (pos[m.POS[1]], -v if target_first else v)])
                    for q in QUESTIONS:
                        s = probe_q(r, q, steer_fn)
                        rows_c.append(dict(set=name, alpha=alpha, kind=kind, vec=vi, question=q,
                                           towards_pushed=sgn * (s - base[q]),
                                           picks_pushed_after=(s > 0) == target_first,
                                           picked_pushed_before=(base[q] > 0) == target_first))
ctrl = pd.DataFrame(rows_c)
tab = (ctrl.groupby(["set", "alpha", "kind", "question"])
           .agg(towards_pushed=("towards_pushed", "mean"), picks_pushed=("picks_pushed_after", "mean"))
           .unstack("question").round(2))
display(tab)

def verdict_for(name, alpha=4):
    g = ctrl[(ctrl.set == name) & (ctrl.alpha == alpha) & (ctrl.kind == "role")].groupby("question").towards_pushed.mean()
    rnd = ctrl[(ctrl.set == name) & (ctrl.alpha == alpha) & (ctrl.kind == "random")].groupby("question").towards_pushed.mean()
    cause, effect, first = g["cause"], g["effect"], g["first"]
    if cause <= 0:
        return "no effect on the cause question", g, rnd
    if effect < -0.5 * cause and abs(first) < 0.25 * cause:
        return "CAUSAL ROLE: the effect answer moves the opposite way and word order is untouched", g, rnd
    if effect > 0.5 * cause and first > 0.5 * cause:
        return "ANSWER SELECTION: the pushed option is picked whatever the question", g, rnd
    return "MIXED: partly role, partly answer selection", g, rnd

print("Mean shift towards the pushed noun at strength 4 (logits; random directions in brackets):")
ctrl_verdicts = {}
for name in ctrl_sets:
    v, g, rnd = verdict_for(name)
    ctrl_verdicts[name] = v
    print(f"  {name}\n     cause {g['cause']:+.2f} ({rnd['cause']:+.2f})   effect {g['effect']:+.2f} ({rnd['effect']:+.2f})   "
          f"first word {g['first']:+.2f} ({rnd['first']:+.2f})\n     -> {v}")
try:
    ctrl.to_csv(f"{out_dir}/control_questions.csv"); print("saved to", out_dir)
except Exception:
    pass

## 14. Remove the answer-selection signal, then retest
A pure "pick this option" direction is estimated from the first-word question, where causal role is irrelevant, and projected out of the Arabic and English role directions. Readout, the three-question steering control and the cross-language cosine are rerun with the cleaned directions.

In [ ]:
# ===== 14. Remove the answer-selection signal, then retest role =====
# 1) Estimate a pure "pick this option" direction from the FIRST-WORD question, where causal role is irrelevant.
# 2) Project it out of the Arabic (knowledge) and English (grammar) role directions.
# 3) Rerun readout, the three-question steering control, and the cosine with the cleaned directions.
def clean(u, s):
    u = u - (u @ s) * s
    return u / np.linalg.norm(u)

@torch.no_grad()
def option_states_q(r, q):
    """States at the two option nouns for question q, in both option orders, at layer LSTAR.
    Returns (score, [(first-noun option, second-noun option) for each order]); score > 0 = picks the first noun."""
    lang = "Arabic" if r["lang"] == "ar" else "English"
    a, b = r.get("first_tok", r["first"]), r.get("second_tok", r["second"])
    ds, pairs_ = [], []
    for o1, o2, flipped in ((r["first"], r["second"], False), (r["second"], r["first"], True)):
        t = prompt_q(r["sentence"], o1, o2, lang, q)
        ids, p = encode(t, r["sentence"], a, b, o1, o2)
        out = model(input_ids=ids, output_hidden_states=True)
        lp = torch.log_softmax(out.logits[0, -1].float(), -1)
        ds.append((lp[A_ID] - lp[B_ID]).item())
        h = out.hidden_states[LSTAR][0]
        hA, hB = h[p[3]].float().cpu().numpy(), h[p[4]].float().cpu().numpy()
        pairs_.append((hB, hA) if flipped else (hA, hB))
    return (ds[0] - ds[1]) / 2, pairs_

# ---- 1. selection direction from the first-word question ----
sel_items = [r for r in recs] + [r for r in en_recs]
sel_diffs, sel_meta = [], []
for r in sel_items:
    sc, prs = option_states_q(r, "first")
    if abs(sc) < 1.0:
        continue                                   # only confident picks
    d = np.mean([f - s for f, s in prs], axis=0)   # first-noun option minus second-noun option, averaged over orders
    sel_diffs.append(d if sc > 0 else -d)          # picked minus not picked
    sel_meta.append(r)
sel_diffs = np.array(sel_diffs)
s_dir = m.unit(sel_diffs.mean(0))
half = len(sel_diffs) // 2                          # sanity check: a direction from one half predicts picks in the other
s_half = m.unit(sel_diffs[:half].mean(0))
print(f"Selection direction from {len(sel_diffs)} confident first-word answers; "
      f"held-out half predicts the pick in {np.mean(sel_diffs[half:] @ s_half > 0):.0%} of items")

# ---- 2. clean the role directions ----
u_ar_raw, _ = m.direction(H[tr_idx], [recs[i] for i in tr_idx], know, LSTAR)
u_en_raw, _ = m.direction(en_H, en_recs, gramr, LSTAR)
u_ar_c, u_en_c = clean(u_ar_raw, s_dir), clean(u_en_raw, s_dir)
print(f"cos(Arabic role, selection) {u_ar_raw @ s_dir:+.3f} | cos(English role, selection) {u_en_raw @ s_dir:+.3f}")
print(f"cos(Arabic, English) raw {u_ar_raw @ u_en_raw:+.3f} -> cleaned {u_ar_c @ u_en_c:+.3f}")

# ---- 3. readout with cleaned directions ----
def lopo_dir(Hx, rx, mask, ref, s=None, subset=None):
    hits = []
    for p in sorted({r["pair"] for r, k in zip(rx, mask) if k}):
        tr = [i for i, (r, k) in enumerate(zip(rx, mask)) if k and r["pair"] != p]
        te = [i for i, (r, k) in enumerate(zip(rx, mask)) if k and r["pair"] == p and (subset is None or subset(r))]
        if not te: continue
        u, _ = m.direction(Hx[tr], [rx[i] for i in tr], ref, LSTAR)
        if s is not None: u = clean(u, s)
        d, _ = m.role_diffs(Hx[te], [rx[i] for i in te], ref, LSTAR)
        hits += list(d @ u > 0)
    return float(np.mean(hits)) if hits else np.nan

read_sets = {
 "Arabic unvocalised familiar (vs knowledge)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"].startswith("U")), know),
 "  pairs with a role-switching word": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"].startswith("U") and r["switch"]), know),
 "Arabic, grammar contradicts knowledge (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "familiar" and r["kind"] in ("V_vos", "V_nom_xy")), gramr),
 "Arabic 'resulted from' (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["kind"] == "N_res"), gramr),
 "Arabic vocalised fictional (vs grammar)": (subset_idx(lambda r: r["lang"] == "ar" and r["set"] == "fictional" and r["kind"].startswith("V")), gramr),
 "English fictional (vs grammar)": (subset_idx(lambda r: r["lang"] == "en" and r["set"] == "fictional"), gramr),
}
rows_r = []
for name, (idx, ref) in read_sets.items():
    sub = [recs[i] for i in idx]
    row = dict(items=name)
    for label, u in (("Arabic dir, raw", u_ar_raw), ("Arabic dir, cleaned", u_ar_c),
                     ("English dir, raw", u_en_raw), ("English dir, cleaned", u_en_c)):
        row[label] = m.paired_acc(H[idx], sub, ref, LSTAR, u)[0]
    rows_r.append(row)
read_tab = pd.DataFrame(rows_r).set_index("items")
read_tab.loc["Arabic unvocalised familiar, HELD-OUT pairs (vs knowledge)"] = [
    lopo_dir(H, recs, is_ar_unvoc_fam, know), lopo_dir(H, recs, is_ar_unvoc_fam, know, s_dir), np.nan, np.nan]
display(read_tab.round(2))

# ---- 4. three-question steering with the cleaned Arabic direction ----
d_tr, _ = m.role_diffs(H[tr_idx], [recs[i] for i in tr_idx], know, LSTAR)
gap_c = float((d_tr @ u_ar_c).mean())
UC = torch.tensor(u_ar_c, dtype=torch.bfloat16, device=model.device)
rows_s = []
for name, items in steer_sets.items():
    for r in items:
        target_first = r["score"] < 0
        sgn = 1 if target_first else -1
        base = {q: probe_q(r, q) for q in QUESTIONS}
        for alpha in (4, 8):
            for kind, vecs in (("cleaned role", [UC]), ("random", RAND[:2])):
                for vi, vec in enumerate(vecs):
                    def steer_fn(pos, vec=vec, alpha=alpha):
                        v = vec * (alpha * gap_c / 2)
                        return (LSTAR, [(pos[m.POS[0]], v if target_first else -v), (pos[m.POS[1]], -v if target_first else v)])
                    for q in QUESTIONS:
                        rows_s.append(dict(set=name, alpha=alpha, kind=kind, question=q,
                                           towards_pushed=sgn * (probe_q(r, q, steer_fn) - base[q])))
cs = pd.DataFrame(rows_s)
print("\nShift towards the pushed noun with the CLEANED direction (logits; random in brackets)")
clean_verdicts = {}
for name in steer_sets:
    for alpha in (4, 8):
        g = cs[(cs.set == name) & (cs.alpha == alpha) & (cs.kind == "cleaned role")].groupby("question").towards_pushed.mean()
        rd = cs[(cs.set == name) & (cs.alpha == alpha) & (cs.kind == "random")].groupby("question").towards_pushed.mean()
        c, e, f = g["cause"], g["effect"], g["first"]
        if c <= 0:
            v = "no effect"
        elif e < -0.5 * c and abs(f) < 0.25 * c:
            v = "CAUSAL ROLE"
        elif e > 0.5 * c and f > 0.5 * c:
            v = "ANSWER SELECTION"
        else:
            v = "MIXED"
        clean_verdicts[(name, alpha)] = v
        print(f"  {name}, strength {alpha}\n     cause {c:+.2f} ({rd['cause']:+.2f})   effect {e:+.2f} ({rd['effect']:+.2f})   "
              f"first word {f:+.2f} ({rd['first']:+.2f})   role estimates: {c - f:+.2f} / {f - e:+.2f}   -> {v}")
try:
    read_tab.to_csv(f"{out_dir}/cleaned_readout.csv"); cs.to_csv(f"{out_dir}/cleaned_steering.csv"); print("saved to", out_dir)
except Exception:
    pass


## 15. Patching inside the sentence
States from "X caused Y" (VSO) are patched into "Y caused X" (VOS) at both nouns or at the closing quote, one layer at a time. If the information is used as a role, the cause answer moves towards X while the effect answer moves away from X.

In [ ]:
# ===== 15. Patching inside the sentence: is the role USED as a role? =====
# Base: "Y caused X" (سَبَّبَ Xَ Yُ). Source: "X caused Y" (سَبَّبَ Xُ Yَ). Same words, same order, case endings swapped.
# Copy source states into the base, one layer at a time, at (a) both nouns or (b) the closing quote.
# If the patched information is used as a ROLE: cause answer moves to X, effect answer moves to Y.
PATCH_LAYERS = list(range(4, L, 4))
WHERE = {"both nouns": [0, 1], "closing quote": [2]}

@torch.no_grad()
def prep(sentence, a, b, x, y, q):
    outs = []
    for o1, o2 in ((x, y), (y, x)):
        t = prompt_q(sentence, o1, o2, "Arabic", q)
        ids, p = encode(t, sentence, a, b, o1, o2)
        out = model(input_ids=ids, output_hidden_states=True)
        lp = torch.log_softmax(out.logits[0, -1].float(), -1)
        hs = torch.stack([h[0, p[:3], :] for h in out.hidden_states])     # [L+1, 3, d] at noun 1, noun 2, quote
        outs.append(dict(ids=ids, pos=p, hs=hs, d=(lp[A_ID] - lp[B_ID]).item()))
    return outs

score_x = lambda d1, d2: (d1 - d2) / 2          # > 0: the model picks X (the first noun)
by_kind = {(r["pair"], r["kind"]): r for r in recs if r["lang"] == "ar"}
pairs_p = sorted({r["pair"] for r in recs if r["lang"] == "ar"})
rows_p = []
for pid in pairs_p:
    rb, rs = by_kind[(pid, "V_vos")], by_kind[(pid, "V_vso")]      # base: Y caused X ; source: X caused Y
    x, y = rb["first"], rb["second"]
    for q in QUESTIONS:
        B = prep(rb["sentence"], rb["first_tok"], rb["second_tok"], x, y, q)
        S = prep(rs["sentence"], rs["first_tok"], rs["second_tok"], x, y, q)
        base = score_x(B[0]["d"], B[1]["d"])
        for l in PATCH_LAYERS:
            for wh, idxs in WHERE.items():
                ds = []
                for k in (0, 1):
                    edits = [(B[k]["pos"][i], S[k]["hs"][l, i] - B[k]["hs"][l, i]) for i in idxs]
                    ds.append(ab_logit(B[k]["ids"], (l, edits)))
                rows_p.append(dict(pair=pid, set=rb["set"], question=q, layer=l, site=wh,
                                   delta=score_x(*ds) - base))
pt = pd.DataFrame(rows_p)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, wh in zip(axes, WHERE):
    g = pt[pt["site"] == wh].groupby(["layer", "question"])["delta"].mean().unstack()
    for q in QUESTIONS:
        ax.plot(g.index, g[q], lw=2, label={"cause": "which is the cause? (want +)", "effect": "which is the effect? (want -)",
                                            "first": "which word comes first?"}[q])
    ax.axhline(0, color="grey", ls=":", lw=1); ax.set_title(f"patched at {wh}"); ax.set_xlabel("layer")
axes[0].set_ylabel("shift towards X (logits)"); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Best layer per patch site (largest cause shift):")
for wh in WHERE:
    for setname in ("familiar", "fictional", None):
        sub = pt[(pt["site"] == wh) & ((pt["set"] == setname) if setname else True)]
        g = sub.groupby(["layer", "question"])["delta"].mean().unstack()
        lb = int(g["cause"].idxmax()); c, e, f = g.loc[lb, "cause"], g.loc[lb, "effect"], g.loc[lb, "first"]
        v = ("no effect" if c < 1.0 else "cause and effect move in OPPOSITE directions (role-like)" if e < -0.5 * c
             else "cause and effect move together (selection-like)" if e > 0.5 * c else "mixed")
        print(f"  {wh:13s} | {setname or 'all pairs':9s} | layer {lb:2d}: cause {c:+.2f}  effect {e:+.2f}  first word {f:+.2f}  -> {v}")
try:
    pt.to_csv(f"{out_dir}/patching_in_sentence.csv"); print("saved to", out_dir)
except Exception:
    pass


## 16. Word-order check

In [ ]:
# ===== 16. Why did the word-order answer move? Does the model answer "which word comes first?" by position? =====
# If the model gets word order wrong when the grammatical subject comes second (VOS), the first-word question
# is not role-neutral for these sentences, and its shift under patching is expected.
fw = []
for pid in pairs_p:
    for kind in ("U_xy", "V_vso", "V_vos"):
        r = by_kind[(pid, kind)]
        fw.append(dict(kind=kind, picks_first_noun=probe_q(r, "first") > 0))
fw = pd.DataFrame(fw)
print("Share of items where the model says the FIRST noun comes first:")
print(fw.groupby("kind").picks_first_noun.mean().round(2))
try:
    fw.to_csv(f"{out_dir}/first_word_check.csv"); print("saved to", out_dir)
except Exception:
    pass
